# KAIROS: NIFTY 50 Market Movement Analysis
UML501 — Machine Learning

Data acquisition, preprocessing, exploratory analysis, candidate predictors and classification targets.

**Execution:** Place the two source CSV files in the working directory or set `KAIROS_DATA_DIR` to their location. The optional acquisition function is not executed by default.

**Development assistance:** AI-assisted coding and editing were used in preparing this notebook; the methodology and results should be independently reviewed.

## 1. Problem statement
Investigate whether historical NIFTY 50 prices and India VIX can predict significant net price movements over the next five trading sessions. Forecasts are issued after market close. The provisional threshold is ±3%.

## 2. Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import os

## 3. Data collection
### 3.1 Source and collection method
Historical NIFTY 50 observations were obtained from NSE Indices. The optional acquisition function below reconstructs the original batched HTTP request procedure; the live endpoint has not been retested. Fixed CSV snapshots are used for reproducibility. The original India VIX acquisition script is unavailable.
### 3.2 Optional NIFTY acquisition code

In [ ]:
def collect_nifty_history(start_date, end_date):
    import requests
    import json
    import time
    from datetime import timedelta

    url = 'https://www.niftyindices.com/BackPage/getHistoricaldatatabletoString'
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Content-Type': 'application/json',
        'X-Requested-With': 'XMLHttpRequest',
        'Referer': 'https://www.niftyindices.com/reports/historical-data',
        'Origin': 'https://www.niftyindices.com'
    }
    if start_date > end_date:
        raise ValueError('Start date must precede end date.')
    frames = []
    with requests.Session() as session:
        session.headers.update(headers)
        current_start = start_date
        while current_start <= end_date:
            current_end = min(current_start + timedelta(days=364), end_date)
            cinfo = ("{'name':'NIFTY 50','startDate':'" + current_start.strftime('%m/%d/%Y')
                     + "','endDate':'" + current_end.strftime('%m/%d/%Y')
                     + "','indexName':'NIFTY 50'}")
            for attempt in range(3):
                try:
                    response = session.post(url, json={'cinfo': cinfo}, timeout=90)
                    response.raise_for_status()
                    result = response.json()
                    if isinstance(result, dict):
                        result = result['d']
                    if isinstance(result, str):
                        result = json.loads(result)
                    if not isinstance(result, list):
                        raise ValueError('Unexpected response format')
                    if result:
                        frames.append(pd.DataFrame(result))
                    elif (current_end - current_start).days >= 7:
                        raise ValueError('Unexpected empty multi-day batch')
                    break
                except (requests.RequestException, ValueError, KeyError) as exc:
                    if attempt == 2:
                        raise RuntimeError(f'Failed batch {current_start} to {current_end}') from exc
                    time.sleep(2 ** attempt)
            current_start = current_end + timedelta(days=1)
            time.sleep(1.5)
    if not frames:
        raise ValueError('No observations returned')
    result = pd.concat(frames, ignore_index=True).rename(columns={
        'HistoricalDate': 'Date', 'date': 'Date', 'OPEN': 'Open',
        'HIGH': 'High', 'LOW': 'Low', 'CLOSE': 'Close'
    })[['Date', 'Open', 'High', 'Low', 'Close']]
    result['Date'] = pd.to_datetime(result['Date'], errors='raise')
    for col in ['Open', 'High', 'Low', 'Close']:
        result[col] = pd.to_numeric(result[col], errors='raise')
    return result.sort_values('Date').reset_index(drop=True)

# Optional historical request (not run):
# from datetime import date
# collected = collect_nifty_history(date(2006, 11, 1), date(2026, 11, 14))
# collected.to_csv('NIFTY_RECOLLECTED.csv', index=False)

### 3.3 Load source datasets
The source CSV files remain unchanged throughout the analysis.

In [ ]:
base = Path(os.environ.get('KAIROS_DATA_DIR', str(Path.cwd())))
nifty_file = base / 'KAIROS_NIFTY50_NSE_2006_2026.csv'
vix_file = base / 'KAIROS_INDIA_VIX_2008_2026.csv'
for file in (nifty_file, vix_file):
    if not file.is_file():
        raise FileNotFoundError(f'Upload {file.name} to {base} before continuing.')
nifty = pd.read_csv(nifty_file)
vix = pd.read_csv(vix_file)
print('NIFTY:', nifty.shape)
print('VIX:', vix.shape)
display(nifty.head(), vix.head())

## 4. Data preprocessing
### 4.1 Check column names and data types

In [ ]:
prices = ['Open', 'High', 'Low', 'Close']
assert set(nifty.columns) == set(['Date'] + prices)
assert set(vix.columns) == {'Date', 'VIX_Close'}
print(nifty.dtypes)
print(vix.dtypes)

### 4.2 Convert dates and sort chronologically

In [ ]:
for frame in (nifty, vix):
    frame['Date'] = pd.to_datetime(frame['Date'], errors='raise')
    frame.sort_values('Date', inplace=True)
    frame.reset_index(drop=True, inplace=True)
    assert frame.Date.notna().all()
    assert frame.Date.is_monotonic_increasing
print('NIFTY:', nifty.Date.min().date(), 'to', nifty.Date.max().date())
print('VIX:', vix.Date.min().date(), 'to', vix.Date.max().date())

### 4.3 Check missing values

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'missing values:')
    print(frame.isna().sum())
    assert frame.notna().all().all()

### 4.4 Check duplicate dates

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'duplicate dates:', frame.Date.duplicated().sum())
    assert frame.Date.is_unique

### 4.5 Validate OHLC and VIX values

In [ ]:
assert np.isfinite(nifty[prices]).all().all()
assert (nifty[prices] > 0).all().all()
assert (nifty.High >= nifty[['Open', 'Close', 'Low']].max(axis=1)).all()
assert (nifty.Low <= nifty[['Open', 'Close', 'High']].min(axis=1)).all()
assert np.isfinite(vix.VIX_Close).all() and (vix.VIX_Close > 0).all()
print('OHLC and VIX checks passed.')

### 4.6 Align the date ranges
Use the common observation period. Earlier NIFTY records are excluded from the joint experiment.

In [ ]:
start = max(nifty.Date.min(), vix.Date.min())
end = min(nifty.Date.max(), vix.Date.max())
assert start <= end
aligned = nifty.loc[nifty.Date.between(start, end)].copy()
vix_aligned = vix.loc[vix.Date.between(start, end)].copy()
assert vix_aligned.Date.isin(aligned.Date).all()
print('Common period:', start.date(), 'to', end.date())
print('Excluded NIFTY rows:', len(nifty) - len(aligned))
print('Aligned NIFTY rows:', len(aligned))
print('Aligned VIX rows:', len(vix_aligned))

### 4.7 Merge the datasets
Use a one-to-one left join to preserve aligned NIFTY sessions.

In [ ]:
df = aligned.merge(vix_aligned, on='Date', how='left', validate='one_to_one')
assert len(df) == len(aligned)
assert df.Date.is_unique and df.Date.is_monotonic_increasing
print('Merged shape before missing-value flag:', df.shape)
print('Missing VIX observations:', df.VIX_Close.isna().sum())
display(df.loc[df.VIX_Close.isna()])

### 4.8 Handle missing VIX observations
Flag missing readings before carrying forward the last available earlier value. A carried-forward reading is not an observed value.

In [ ]:
df['VIX_missing'] = df.VIX_Close.isna().astype(int)
print('Missing VIX dates:', df.loc[df.VIX_missing.eq(1), 'Date'].dt.date.tolist())
df['VIX_Close'] = df.VIX_Close.ffill()
assert df.notna().all().all()
assert len(df) == len(aligned)
print('Remaining missing values:', df.isna().sum().sum())
print('Cleaned dataset shape:', df.shape)
display(df.head())

### 4.9 Export the cleaned combined dataset
The market-data export excludes the internal imputation flag. A separate audit file records the affected dates and carried-forward VIX values. The internal flag is retained for feature-quality checks.

In [ ]:
output_path = base / 'KAIROS_COMBINED_CLEANED.csv'
audit_path = base / 'KAIROS_VIX_IMPUTATION_AUDIT.csv'
combined = df.drop(columns=['VIX_missing'])
combined.to_csv(output_path, index=False)
df.loc[df.VIX_missing.eq(1), ['Date', 'VIX_Close']].to_csv(audit_path, index=False)
print('Combined dataset:', combined.shape)
print('Imputation audit:', audit_path.name)
# from google.colab import files
# files.download(str(output_path))

## 5. Chronological experiment boundaries
Use 70% for training, 15% for validation and 15% for testing. EDA uses the training period only. The test partition is a procedural holdout rather than an independent external dataset because the historical snapshot was previously examined.

In [ ]:
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)
train = df.iloc[:train_end].copy()
for name, part in [('Train', train), ('Validation', df.iloc[train_end:val_end]), ('Test', df.iloc[val_end:])]:
    print(name, len(part), part.Date.min().date(), part.Date.max().date())

## 6. Exploratory Data Analysis (EDA)
### 6.1 Descriptive statistics

In [ ]:
display(train[prices + ['VIX_Close']].describe().round(2))

### 6.2 Market trends
NIFTY closing levels and implied volatility are plotted separately.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Close, linewidth=1)
axes[0].set(title='NIFTY 50 closing level', ylabel='Index points')
axes[1].plot(train.Date, train.VIX_Close, linewidth=1)
axes[1].set(title='India VIX', ylabel='Index level', xlabel='Date')
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### 6.3 Returns and rolling volatility
Compute daily percentage returns and trailing 20-session standard deviation.

In [ ]:
train['Return_1d'] = train.Close.pct_change() * 100
train['Volatility_20d'] = train.Return_1d.rolling(20).std()
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Return_1d, linewidth=0.7)
axes[0].set(title='Daily NIFTY returns', ylabel='Return (%)')
axes[1].plot(train.Date, train.Volatility_20d, linewidth=1)
axes[1].set(title='20-session rolling volatility', ylabel='Daily std. dev. (%)', xlabel='Date')
fig.tight_layout()
plt.show()
print(train.Return_1d.describe().round(3))
print('Largest daily decline:', train.loc[train.Return_1d.idxmin(), ['Date','Return_1d']].to_dict())
print('Largest daily rise:', train.loc[train.Return_1d.idxmax(), ['Date','Return_1d']].to_dict())

### 6.4 VIX and market returns
Pearson correlation measures linear association, not predictive performance.

In [ ]:
train['VIX_Change'] = train.VIX_Close.pct_change() * 100
train['Abs_Return'] = train.Return_1d.abs()
cols = ['Return_1d', 'Abs_Return', 'VIX_Close', 'VIX_Change', 'Volatility_20d']
corr = train[cols].corr()
display(corr.round(3))
plt.figure(figsize=(7, 4))
plt.scatter(train.VIX_Change, train.Return_1d, s=8, alpha=0.35)
plt.axhline(0, linewidth=0.7)
plt.axvline(0, linewidth=0.7)
plt.xlabel('Daily VIX change (%)')
plt.ylabel('Daily NIFTY return (%)')
plt.title('Same-session return and VIX change')
plt.tight_layout()
plt.show()

### 6.5 Candidate target distribution
Use a provisional five-session return threshold of ±3%. Exclude unknown outcomes.

In [ ]:
forward = (train.Close.shift(-5) / train.Close - 1) * 100
observed = forward.dropna()
labels = pd.Series(np.select([observed < -3, observed > 3], ['Major Down', 'Major Up'], default='Ordinary'), index=observed.index)
order = ['Major Down', 'Ordinary', 'Major Up']
counts = labels.value_counts().reindex(order, fill_value=0)
summary = pd.DataFrame({'Count': counts, 'Percent': counts / len(labels) * 100})
display(summary.round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(observed, bins=60, edgecolor='white')
for boundary in (-3, 3):
    axes[0].axvline(boundary, linestyle='--', linewidth=1.4)
axes[0].set(title='Five-session forward returns', xlabel='Return (%)', ylabel='Observations')
axes[1].bar(order, counts)
axes[1].set(title='Candidate target distribution', ylabel='Observations')
fig.tight_layout()
plt.show()
assert len(labels) == len(train) - 5
print('Excluded rows without a within-training outcome:', len(train) - len(labels))

### 6.6 EDA conclusions
Class imbalance requires macro F1 and per-class precision and recall alongside accuracy. Same-session correlations are not evidence of future forecasting skill.

## 7. Feature engineering
### 7.1 Candidate predictors
Forecasts are issued after market close. Trailing features may use earlier history across partition boundaries. The imputation flag is retained for data-quality checks only, not as a predictor. The first 20 sessions are warm-up rows.

In [ ]:
def make_features(data):
    features = pd.DataFrame(index=data.index)
    close = data['Close']
    vix_close = data['VIX_Close']
    daily_return = close.pct_change(fill_method=None) * 100
    features['Return_1d'] = daily_return
    features['Return_5d'] = close.pct_change(5, fill_method=None) * 100
    features['Volatility_20d'] = daily_return.rolling(20).std()
    features['MA_20_gap_pct'] = (close / close.rolling(20).mean() - 1) * 100
    features['Intraday_range_pct'] = (data['High'] - data['Low']) / data['Open'] * 100
    features['VIX_Close'] = vix_close
    features['VIX_change_1d'] = vix_close.pct_change(fill_method=None) * 100
    # Mark changes involving an imputed VIX as missing rather than inventing a move.
    unreliable = data['VIX_missing'].eq(1) | data['VIX_missing'].shift(1, fill_value=0).eq(1)
    features.loc[unreliable, 'VIX_change_1d'] = np.nan
    return features

X = make_features(df)
assert X.shape[1] == 7 and "VIX_missing" not in X.columns
X_train = X.iloc[:train_end].copy()
print('Candidate predictors:', X_train.shape[1])
print('Training rows with all predictors available:', X_train.dropna().shape[0])
display(X_train.iloc[20:25].round(3))

### 7.2 Feature validation and leakage checks
Prefix invariance tests for accidental use of future observations.

In [ ]:
for cutoff in (50, 250, train_end, val_end):
    pd.testing.assert_frame_equal(make_features(df.iloc[:cutoff]), X.iloc[:cutoff])
assert not np.isinf(X.to_numpy(dtype=float)).any()
assert X.index.equals(df.index)
print('Prefix-invariance, finite-value and alignment checks passed.')
print('Training missing counts:')
print(X_train.isna().sum())

## 8. Target variable creation
### 8.1 Five-session classification labels
Returns above +3% map to 1, below −3% to −1, and the inclusive interval to 0. A numerical tolerance protects threshold boundaries. Outcomes are generated separately in each partition.

In [ ]:
"""Five-session classification labels, with partition boundary protection."""
import numpy as np
import pandas as pd


def make_targets(data, horizon=5, threshold=3.0):
    if horizon < 1 or threshold <= 0:
        raise ValueError('horizon and threshold must be positive')
    if not data.Date.is_monotonic_increasing or not data.Date.is_unique:
        raise ValueError('Dates must be unique and sorted')
    forward = (data.Close.shift(-horizon) / data.Close - 1) * 100
    boundary = np.isclose(forward.abs(), threshold, rtol=0, atol=1e-10)
    target = pd.Series(pd.NA, index=data.index, dtype='Int64')
    target.loc[forward.notna()] = 0
    target.loc[(forward > threshold) & ~boundary] = 1
    target.loc[(forward < -threshold) & ~boundary] = -1
    return pd.DataFrame({'Outcome_date': data.Date.shift(-horizon),
                         'Forward_return_5d_pct': forward, 'Target': target})


### 8.2 Prepare aligned training observations
Forward returns and outcome dates are retained for evaluation only. Exclude unavailable features and unknown outcomes.

In [ ]:
partitions = {'train': slice(0, train_end), 'validation': slice(train_end, val_end), 'test': slice(val_end, len(df))}
feature_columns = X.columns.tolist()
training_labels = make_targets(df.iloc[partitions['train']])
training_mask = X.iloc[partitions['train']].notna().all(axis=1) & training_labels.Target.notna()
X_fit = X.iloc[partitions['train']].loc[training_mask, feature_columns].copy()
y_fit = training_labels.loc[training_mask, 'Target'].astype(int)
print('Eligible training observations:', len(y_fit))
print('Excluded for unknown outcomes:', training_labels.Target.isna().sum())
print('Excluded for unavailable predictors among labelled rows:', ((~X.iloc[partitions['train']].notna().all(axis=1)) & training_labels.Target.notna()).sum())
counts_fit = y_fit.value_counts().reindex([-1, 0, 1], fill_value=0)
print(pd.DataFrame({'Count': counts_fit, 'Percent': (counts_fit / len(y_fit) * 100).round(2)}))

### 8.3 Validate target boundaries and alignment

In [ ]:
for future, expected in [(96, -1), (97, 0), (100, 0), (103, 0), (104, 1)]:
    example = pd.DataFrame({'Date': pd.date_range('2020-01-01', periods=6), 'Close': [100]*5 + [future]})
    check = make_targets(example)
    assert check.Target.iloc[0] == expected
    assert check.Target.iloc[1:].isna().all()
for part in partitions.values():
    block = df.iloc[part]
    result = make_targets(block)
    assert result.Target.tail(5).isna().all()
    assert result.Outcome_date.dropna().le(block.Date.max()).all()
    assert result.Target.notna().sum() == len(block) - 5
assert X_fit.index.equals(y_fit.index)
assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct'}.intersection(X_fit.columns)
assert X_fit.notna().all().all()
print('Target threshold, partition boundary and feature alignment checks passed.')

## 9. Final dataset preparation
### 9.1 Training feature review
Review training-period missingness, variability and pairwise correlations. Correlation screening is descriptive; feature selection is deferred to chronological model validation.

In [ ]:
# Review training features only; do not use validation or test correlations.
review = pd.DataFrame({'Missing': X_train.isna().sum(),
                       'Unique_values': X_train.nunique(),
                       'Std_dev': X_train.std()})
display(review.round(4))
assert X_train.nunique().gt(1).all()
correlations = X_train.corr()
pairs = correlations.where(np.triu(np.ones(correlations.shape, dtype=bool), k=1)).stack()
strong_pairs = pairs[pairs.abs() >= 0.80].sort_values(key=abs, ascending=False)
print('Feature correlations with |r| >= 0.80:')
print(strong_pairs.round(3) if len(strong_pairs) else 'None')

### 9.2 Assemble partition-safe datasets
Only observations with all seven features and a known outcome within their chronological partition are eligible. The research table contains market values, features, outcome metadata and partition identifiers. Only the explicit feature allowlist is used as model input.

In [ ]:
"""Assemble partition-safe research datasets from aligned prices and features."""
import pandas as pd
import numpy as np

FEATURE_COLUMNS = ['Return_1d', 'Return_5d', 'Volatility_20d', 'MA_20_gap_pct',
                   'Intraday_range_pct', 'VIX_Close', 'VIX_change_1d']
MARKET_COLUMNS = ['Date', 'Open', 'High', 'Low', 'Close', 'VIX_Close']


def build_datasets(df, X, partitions, make_targets):
    assert df.index.is_unique and X.index.equals(df.index)
    assert df.Date.is_unique and df.Date.is_monotonic_increasing
    assert X.columns.tolist() == FEATURE_COLUMNS
    pd.testing.assert_series_equal(X.VIX_Close, df.VIX_Close)
    assert not np.isinf(X.to_numpy(dtype=float)).any()
    assert list(partitions) == ['train', 'validation', 'test']
    used = []
    datasets = {}
    audit = []
    for name, section in partitions.items():
        market = df.iloc[section]
        features = X.iloc[section]
        used.extend(market.index.tolist())
        labels = make_targets(market)
        assert labels.index.equals(market.index)
        feature_ok = features.notna().all(axis=1)
        target_ok = labels.Target.notna()
        eligible = feature_ok & target_ok
        # VIX_Close is already present among the market columns.
        joined = market[MARKET_COLUMNS].join(features.drop(columns='VIX_Close')).join(labels)
        joined['Partition'] = name
        part = joined.loc[eligible].copy()
        part['Target'] = part.Target.astype(int)
        assert not part.empty and part.notna().all().all()
        assert part.Target.isin([-1, 0, 1]).all()
        assert part.Outcome_date.gt(part.Date).all()
        assert part.Outcome_date.le(market.Date.max()).all()
        assert part.Date.is_unique and part.Date.is_monotonic_increasing
        datasets[name] = part
        audit.append({'Partition': name, 'Original': len(market),
                      'Unavailable_target': int((~target_ok).sum()),
                      'Unavailable_features_with_target': int((target_ok & ~feature_ok).sum()),
                      'Eligible': len(part)})
    assert used == df.index.tolist()
    final = pd.concat(datasets.values(), ignore_index=True)
    assert final.Date.is_unique and final.Date.is_monotonic_increasing
    assert len(final) == sum(len(part) for part in datasets.values())
    for left, right in [('train', 'validation'), ('validation', 'test')]:
        assert datasets[left].Outcome_date.max() < datasets[right].Date.min()
    # Prediction inputs: latest five sessions with complete features but unknown outcomes.
    recent = df.tail(5)
    recent_features = X.loc[recent.index]
    prediction = recent[['Date']].join(recent_features).dropna().copy()
    assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct'}.intersection(prediction.columns)
    assert set(prediction.Date).isdisjoint(final.Date)
    return final, datasets, prediction, pd.DataFrame(audit)

### 9.3 Validate and summarize the eligible observations
Record exclusions separately for unavailable outcomes and unavailable features. Retain the latest five sessions with complete predictors separately for inference; no future labels are attached.

In [ ]:
final_dataset, datasets, prediction_inputs, dataset_audit = build_datasets(df, X, partitions, make_targets)
print('Dataset preparation audit:')
display(dataset_audit)
print('Final research dataset:', final_dataset.shape)
print('Prediction inputs:', prediction_inputs.shape)

# The model input allowlist is independent of the research CSV schema.
X_fit = datasets['train'][FEATURE_COLUMNS].copy()
y_fit = datasets['train']['Target'].copy()
assert X_fit.index.equals(y_fit.index)
assert len(X_fit) == len(training_labels.loc[training_mask])
assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct', 'Partition'}.intersection(X_fit.columns)
print('Training input shape:', X_fit.shape)
print('Training target shape:', y_fit.shape)

### 9.4 Export reproducible datasets
Export the labelled research table, chronological subsets, inference inputs and an exclusion audit. Source snapshots are not overwritten. Validation and test outcomes are not summarized during preparation.

In [ ]:
export_dir = Path(os.environ.get('KAIROS_OUTPUT_DIR', str(base / 'kairos_datasets')))
export_dir.mkdir(parents=True, exist_ok=True)
exports = {'KAIROS_FINAL_DATASET.csv': final_dataset,
           'KAIROS_TRAIN.csv': datasets['train'],
           'KAIROS_VALIDATION.csv': datasets['validation'],
           'KAIROS_TEST.csv': datasets['test'],
           'KAIROS_PREDICTION_INPUTS.csv': prediction_inputs,
           'KAIROS_DATASET_AUDIT.csv': dataset_audit}
for filename, frame in exports.items():
    path = export_dir / filename
    frame.to_csv(path, index=False)
    saved = pd.read_csv(path)
    assert saved.shape == frame.shape
    assert saved.columns.tolist() == frame.columns.tolist()
    print(filename, saved.shape)

## 10. Baseline models
### 10.1 Benchmark definitions and evaluation metrics
A majority-class classifier and a trailing five-session momentum rule establish reference performance. Both use information available by the prediction date. Accuracy, balanced accuracy, macro F1 and per-class metrics are reported on the validation partition only.

In [ ]:
"""Validation benchmarks for five-session market movement classification."""
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

CLASSES = [-1, 0, 1]
CLASS_NAMES = ['Major Down', 'Ordinary', 'Major Up']


def momentum_predict(X, threshold=3.0):
    """Classify trailing five-session return using the target thresholds."""
    if threshold <= 0:
        raise ValueError('Threshold must be positive')
    returns = X['Return_5d'].to_numpy(dtype=float)
    if not np.isfinite(returns).all():
        raise ValueError('Momentum inputs must be finite')
    boundary = np.isclose(np.abs(returns), threshold, rtol=0, atol=1e-10)
    return np.select([(returns < -threshold) & ~boundary,
                      (returns > threshold) & ~boundary], [-1, 1], default=0)


def evaluate_predictions(y_true, y_pred):
    """Score every target class, including classes never predicted."""
    report = classification_report(y_true, y_pred, labels=CLASSES,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    summary = {'Accuracy': accuracy_score(y_true, y_pred),
               'Balanced accuracy': balanced_accuracy_score(y_true, y_pred),
               'Macro F1': f1_score(y_true, y_pred, labels=CLASSES,
                                    average='macro', zero_division=0),
               'Major Down recall': report['Major Down']['recall'],
               'Major Up recall': report['Major Up']['recall']}
    per_class = pd.DataFrame(report).T.loc[CLASS_NAMES,
                                         ['precision', 'recall', 'f1-score', 'support']]
    matrix = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CLASSES),
                          index=CLASS_NAMES, columns=CLASS_NAMES)
    return summary, per_class, matrix

### 10.2 Fit the majority classifier and generate validation predictions
The majority class is learned from training labels. The momentum rule uses the same fixed ±3% thresholds as the target and does not estimate parameters. Neither uses the validation outcomes to make predictions.

In [ ]:
X_val = datasets['validation'][FEATURE_COLUMNS].copy()
y_val = datasets['validation']['Target'].copy()
assert X_fit.columns.equals(X_val.columns)
assert X_val.index.equals(y_val.index)
assert X_fit.index.intersection(X_val.index).empty
assert set(y_fit.unique()) == set(CLASSES)
assert set(y_val.unique()) == set(CLASSES)

majority = DummyClassifier(strategy='most_frequent')
majority.fit(X_fit, y_fit)
predictions = {'Majority class': majority.predict(X_val),
               'Momentum': momentum_predict(X_val)}
print('Training class counts:')
print(y_fit.value_counts().sort_index())
print('Validation class counts:')
print(y_val.value_counts().sort_index())
assert np.all(predictions['Majority class'] == y_fit.mode().iloc[0])

### 10.3 Validation performance
Confusion matrices use the fixed class order Major Down, Ordinary, Major Up. Precision and recall for classes with no predicted observations are reported as zero where undefined. The test partition is reserved for final model evaluation.

In [ ]:
baseline_rows = []
for name, pred in predictions.items():
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    baseline_rows.append({'Model': name, **metrics})
    print(name, '- validation class metrics')
    display(per_class.round(3))
    print('Confusion matrix (rows: actual; columns: predicted)')
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
baseline_results = pd.DataFrame(baseline_rows).set_index('Model')
display(baseline_results.round(4))

## 11. Multiclass Logistic Regression
### 11.1 Pipeline definition
Standardize predictors using training statistics, then fit a regularized multinomial logistic classifier. Compare unweighted and inverse-frequency class weighting using fixed initial settings (C=1).

In [ ]:
"""Multiclass logistic regression candidates for KAIROS."""
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def build_logistic(class_weight=None):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(solver='lbfgs', C=1.0,
                                          max_iter=2000, class_weight=class_weight))
    ])

### 11.2 Training and validation
Fit each pipeline using training observations only. Evaluate on the same validation observations as the baselines. Check convergence, class ordering and predicted probabilities.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

logistic_models = {
    'Logistic (unweighted)': build_logistic(),
    'Logistic (balanced)': build_logistic(class_weight='balanced')
}
logistic_predictions = {}
logistic_probabilities = {}
logistic_rows = []
for name, model in logistic_models.items():
    with warnings.catch_warnings():
        warnings.simplefilter('error', ConvergenceWarning)
        model.fit(X_fit, y_fit)
    # Standardization statistics must come only from the training data.
    np.testing.assert_allclose(model.named_steps['scaler'].mean_, X_fit.mean().to_numpy())
    assert model.classes_.tolist() == CLASSES
    probabilities = model.predict_proba(X_val)
    pred = model.predict(X_val)
    assert probabilities.shape == (len(X_val), len(CLASSES))
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.array_equal(pred, model.classes_[probabilities.argmax(axis=1)])
    logistic_predictions[name] = pred
    logistic_probabilities[name] = probabilities
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    logistic_rows.append({'Model': name, **metrics})
    print(name)
    display(per_class.round(3))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
logistic_results = pd.DataFrame(logistic_rows).set_index('Model')
comparison = pd.concat([baseline_results, logistic_results])
display(comparison.round(4))

### 11.3 Class weights and probability example
Report inverse-frequency training weights and an illustrative validation observation. No final model selection or test evaluation is performed.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(class_weight='balanced', classes=np.array(CLASSES), y=y_fit.to_numpy())
display(pd.DataFrame({'Class': CLASS_NAMES, 'Training_count': y_fit.value_counts().reindex(CLASSES).to_numpy(),
                      'Balanced_weight': weights}).round(4))
# Show a validation forecast with its corresponding observed outcome.
example = pd.DataFrame({name: probabilities[0] for name, probabilities in logistic_probabilities.items()}, index=CLASS_NAMES)
print('Validation example date:', datasets['validation'].Date.iloc[0].date())
print('Observed label:', y_val.iloc[0])
display(example.round(4))

## 12. Decision trees
### 12.1 Model definition
Gini-based classification trees are fitted without feature scaling. Two depth-limited candidates (maximum depth 4, minimum leaf size 30) compare ordinary and balanced class weighting. An unrestricted unweighted tree provides an overfitting diagnostic.

In [ ]:
"""Decision tree candidates for market movement classification."""
from sklearn.tree import DecisionTreeClassifier


def build_tree(class_weight=None, max_depth=4, min_samples_leaf=30):
    return DecisionTreeClassifier(criterion='gini', max_depth=max_depth,
                                  min_samples_leaf=min_samples_leaf,
                                  class_weight=class_weight, random_state=42)

### 12.2 Training and validation
Fit on training observations only and evaluate on the same validation partition as previous models. Record training macro F1, validation metrics and tree complexity. The unrestricted tree is diagnostic rather than a tuned candidate.

In [ ]:
tree_models = {
    'Tree (unweighted)': build_tree(),
    'Tree (balanced)': build_tree(class_weight='balanced'),
    'Tree (unrestricted)': build_tree(max_depth=None, min_samples_leaf=1)
}
tree_predictions = {}
tree_rows = []
for name, model in tree_models.items():
    model.fit(X_fit, y_fit)
    pred = model.predict(X_val)
    probabilities = model.predict_proba(X_val)
    assert model.classes_.tolist() == CLASSES
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
    assert np.array_equal(pred, model.classes_[probabilities.argmax(axis=1)])
    tree_predictions[name] = pred
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    train_metrics, _, _ = evaluate_predictions(y_fit, model.predict(X_fit))
    tree_rows.append({'Model': name, **metrics,
                      'Train macro F1': train_metrics['Macro F1'],
                      'Depth': model.get_depth(), 'Leaves': model.get_n_leaves()})
    print(name)
    display(per_class.round(3))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
tree_results = pd.DataFrame(tree_rows).set_index('Model')
display(tree_results.round(4))
comparison = pd.concat([baseline_results, logistic_results, tree_results[baseline_results.columns]])
display(comparison.round(4))

### 12.3 Inspect the learned tree
Report the balanced tree root split and complexity, and plot the first three levels. Weighted class frequencies affect split impurity and predicted class scores.

In [ ]:
from sklearn.tree import plot_tree

balanced_tree = tree_models['Tree (balanced)']
root_feature = FEATURE_COLUMNS[balanced_tree.tree_.feature[0]]
root_threshold = balanced_tree.tree_.threshold[0]
print('Balanced tree root:', root_feature, '<=', round(root_threshold, 4))
print('Root Gini impurity:', round(balanced_tree.tree_.impurity[0], 4))
print('Actual depth:', balanced_tree.get_depth())
print('Leaf count:', balanced_tree.get_n_leaves())
assert balanced_tree.get_depth() <= 4
assert balanced_tree.tree_.n_node_samples[balanced_tree.tree_.children_left == -1].min() >= 30
plt.figure(figsize=(15, 5))
plot_tree(balanced_tree, feature_names=FEATURE_COLUMNS, class_names=CLASS_NAMES,
          max_depth=2, filled=True, rounded=False, fontsize=9)
plt.title('Balanced decision tree (first three levels)')
plt.tight_layout()
plt.show()

## 13. Random forest
### 13.1 Model definition
Compare two forests of 200 bootstrap trees with fixed initial depth, minimum leaf size and square-root feature sampling. Only class weighting differs.

In [ ]:
"""Random forest candidates for market movement classification."""
from sklearn.ensemble import RandomForestClassifier


def build_forest(class_weight=None, n_estimators=200, max_depth=6, min_samples_leaf=20):
    return RandomForestClassifier(n_estimators=n_estimators, criterion='gini',
                                  max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                                  max_features='sqrt', bootstrap=True,
                                  class_weight=class_weight, random_state=42, n_jobs=1)

### 13.2 Training and validation
Fit both forests on the training observations and compare with previous candidates on the same validation partition. Record training macro F1 as a complexity diagnostic.

In [ ]:
forest_models = {
    'Forest (unweighted)': build_forest(),
    'Forest (balanced)': build_forest(class_weight='balanced')
}
forest_predictions = {}
forest_rows = []
for name, model in forest_models.items():
    model.fit(X_fit, y_fit)
    pred = model.predict(X_val)
    probabilities = model.predict_proba(X_val)
    assert model.classes_.tolist() == CLASSES
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
    assert np.array_equal(pred, model.classes_[probabilities.argmax(axis=1)])
    assert len(model.estimators_) == model.n_estimators
    forest_predictions[name] = pred
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    train_metrics, _, _ = evaluate_predictions(y_fit, model.predict(X_fit))
    forest_rows.append({'Model': name, **metrics,
                        'Train macro F1': train_metrics['Macro F1']})
    print(name)
    display(per_class.round(3))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
forest_results = pd.DataFrame(forest_rows).set_index('Model')
display(forest_results.round(4))
comparison = pd.concat([baseline_results, logistic_results, tree_results[baseline_results.columns],
                        forest_results[baseline_results.columns]])
display(comparison.round(4))

### 13.3 Probability aggregation and feature importance
Inspect tree diversity and verify probability averaging. Impurity-based importance is descriptive and can be biased by correlated predictors and split opportunities.

In [ ]:
balanced_forest = forest_models['Forest (balanced)']
assert all(tree.get_depth() <= 6 for tree in balanced_forest.estimators_)
assert all(tree.max_features_ == 2 for tree in balanced_forest.estimators_)
print('Number of trees:', len(balanced_forest.estimators_))
print('Candidate features per split:', balanced_forest.estimators_[0].max_features_)
print('Mean tree depth:', np.mean([tree.get_depth() for tree in balanced_forest.estimators_]))
# RandomForest averages individual tree probability vectors.
example = X_val.iloc[[0]]
individual = np.vstack([tree.predict_proba(example.to_numpy())[0]
                        for tree in balanced_forest.estimators_])
np.testing.assert_allclose(individual.mean(axis=0), balanced_forest.predict_proba(example)[0])
print('First validation observation:', datasets['validation'].Date.iloc[0].date())
display(pd.DataFrame(individual[:5], columns=CLASS_NAMES).round(4))
print('Mean across all trees:', individual.mean(axis=0).round(4))
# Impurity-based importance is descriptive, not causal or out-of-sample evidence.
importance = pd.Series(balanced_forest.feature_importances_, index=FEATURE_COLUMNS).sort_values()
display(importance.sort_values(ascending=False).round(4))
importance.plot.barh(figsize=(8, 4), title='Balanced forest: impurity-based feature importance')
plt.xlabel('Normalized impurity reduction')
plt.tight_layout()
plt.show()

## 14. Gradient boosting
### 14.1 Model definition
Compare multiclass gradient boosting candidates with fixed initial settings: 100 stages, learning rate 0.05, depth 2 and minimum leaf size 30. Balanced sample weights are calculated solely from the training labels.

In [ ]:
"""Gradient boosting candidates for KAIROS."""
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight


def build_boosting(n_estimators=100, learning_rate=0.05, max_depth=2, min_samples_leaf=30):
    return GradientBoostingClassifier(loss='log_loss', n_estimators=n_estimators,
                                      learning_rate=learning_rate, max_depth=max_depth,
                                      min_samples_leaf=min_samples_leaf, subsample=1.0,
                                      random_state=42)


def fit_boosting(model, X, y, balanced=False):
    weights = compute_sample_weight('balanced', y) if balanced else None
    return model.fit(X, y, sample_weight=weights)

### 14.2 Training and validation
Train both candidates on the training partition. Compare validation performance against the same baselines and earlier models. Record training macro F1 for diagnostic purposes.

In [ ]:
boost_models = {'Boosting (unweighted)': build_boosting(),
                'Boosting (balanced)': build_boosting()}
boost_predictions = {}
boost_rows = []
for name, model in boost_models.items():
    fit_boosting(model, X_fit, y_fit, balanced=(name == 'Boosting (balanced)'))
    pred = model.predict(X_val)
    proba = model.predict_proba(X_val)
    assert model.classes_.tolist() == CLASSES
    assert model.estimators_.shape == (100, 3)
    np.testing.assert_allclose(proba.sum(axis=1), 1.0)
    assert np.array_equal(pred, model.classes_[proba.argmax(axis=1)])
    boost_predictions[name] = pred
    metrics, per_class, matrix = evaluate_predictions(y_val, pred)
    train_metrics, _, _ = evaluate_predictions(y_fit, model.predict(X_fit))
    boost_rows.append({'Model': name, **metrics, 'Train macro F1': train_metrics['Macro F1']})
    print(name)
    display(per_class.round(4))
    display(matrix)
    assert matrix.to_numpy().sum() == len(y_val)
boost_results = pd.DataFrame(boost_rows).set_index('Model')
display(boost_results.round(4))
comparison = pd.concat([baseline_results, logistic_results, tree_results[baseline_results.columns],
                        forest_results[baseline_results.columns], boost_results[baseline_results.columns]])
print(comparison.sort_values('Macro F1', ascending=False).round(4).to_string())
print(boost_results.round(4).to_string())

### 14.3 Stage diagnostics
Inspect training and validation macro F1 as the ensemble grows. All 100 stages are retained for the initial comparison; stage count is not optimized here.

In [ ]:
from sklearn.metrics import f1_score

balanced_boost = boost_models['Boosting (balanced)']
# Diagnostic only; no validation-based early stopping or stage selection.
stages = np.arange(1, balanced_boost.n_estimators_ + 1)
train_curve = [f1_score(y_fit, pred, labels=CLASSES, average='macro', zero_division=0)
               for pred in balanced_boost.staged_predict(X_fit)]
val_curve = [f1_score(y_val, pred, labels=CLASSES, average='macro', zero_division=0)
             for pred in balanced_boost.staged_predict(X_val)]
np.testing.assert_allclose(val_curve[-1], boost_results.loc['Boosting (balanced)', 'Macro F1'])
assert all(tree.get_depth() <= 2 for tree in balanced_boost.estimators_.ravel())
weights = compute_sample_weight('balanced', y_fit)
print('Total training weight by class:')
print(pd.Series(weights, index=y_fit.index).groupby(y_fit).sum())
print('Stages:', balanced_boost.n_estimators_, '| Trees:', balanced_boost.estimators_.size)
print('Stage diagnostics:')
print(pd.DataFrame({'Stage': stages, 'Training macro F1': train_curve,
                    'Validation macro F1': val_curve}).iloc[[0, 9, 24, 49, 74, 99]].round(4).to_string(index=False))
plt.figure(figsize=(8, 4))
plt.plot(stages, train_curve, label='Training')
plt.plot(stages, val_curve, label='Validation')
plt.xlabel('Boosting stages')
plt.ylabel('Macro F1')
plt.title('Balanced gradient boosting: learning stages')
plt.legend()
plt.tight_layout()
plt.show()

## 15. Hyperparameter tuning
### 15.1 Search definition
Use four expanding-window folds within the training partition, with 400 scored observations per fold and a five-row gap. Confirm that every training outcome precedes the first fold evaluation date. Select one configuration per family by mean fold macro F1. Class weights and scaling are fitted separately in each fold.

In [ ]:
"""Chronological hyperparameter search with fold-local fitting and weighting."""
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.metrics import f1_score

TUNING_GRIDS = {
    'Logistic': {'C': [0.01, 0.1, 1.0, 10.0], 'class_weight': [None, 'balanced']},
    'Tree': {'max_depth': [3, 5, 7], 'min_samples_leaf': [20, 50], 'class_weight': [None, 'balanced']},
    'Forest': {'max_depth': [4, 8], 'min_samples_leaf': [10, 30], 'class_weight': [None, 'balanced']},
    'Boosting': {'n_estimators': [50, 100], 'max_depth': [1, 2], 'balanced': [False, True]}
}


def chronological_folds(training, n_splits=4, test_size=400, gap=5):
    dates = pd.to_datetime(training['Date'])
    outcomes = pd.to_datetime(training['Outcome_date'])
    assert dates.is_unique and dates.is_monotonic_increasing
    splitter = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=gap)
    folds = list(splitter.split(training))
    for tr, va in folds:
        assert tr[-1] + gap < va[0]
        assert outcomes.iloc[tr].max() < dates.iloc[va].min()
        assert set(training.Target.iloc[tr].unique()) == {-1, 0, 1}
        assert set(training.Target.iloc[va].unique()) == {-1, 0, 1}
    return folds


def search_family(name, X, y, folds, fit_candidate):
    rows = []
    for params in ParameterGrid(TUNING_GRIDS[name]):
        scores = []
        for tr, va in folds:
            model = fit_candidate(name, params, X.iloc[tr], y.iloc[tr])
            pred = model.predict(X.iloc[va])
            scores.append(f1_score(y.iloc[va], pred, labels=[-1, 0, 1],
                                   average='macro', zero_division=0))
        rows.append({'Family': name, 'Parameters': params,
                     **{f'Fold {i+1}': score for i, score in enumerate(scores)},
                     'Mean CV macro F1': np.mean(scores), 'SD CV macro F1': np.std(scores)})
    return pd.DataFrame(rows).sort_values('Mean CV macro F1', ascending=False, kind='stable').reset_index(drop=True)

### 15.2 Chronological search
Search a finite grid of initial configurations. For forest and boosting, non-grid settings remain fixed at the previously defined values. Report the top three configurations per family and fold-level variability.

In [ ]:
def fit_candidate(name, params, X, y):
    if name == 'Logistic':
        model = build_logistic(class_weight=params['class_weight'])
        model.set_params(classifier__C=params['C'])
    elif name == 'Tree':
        model = build_tree(**params)
    elif name == 'Forest':
        model = build_forest(**params)
    elif name == 'Boosting':
        settings = {k: v for k, v in params.items() if k != 'balanced'}
        return fit_boosting(build_boosting(**settings), X, y, balanced=params['balanced'])
    else:
        raise ValueError(name)
    return model.fit(X, y)


training = datasets['train']
assert training.index.equals(X_fit.index) and X_fit.index.equals(y_fit.index)
folds = chronological_folds(training)
fold_audit = pd.DataFrame([{'Fold': i+1, 'Train rows': len(tr), 'Gap rows': va[0]-tr[-1]-1,
                            'CV rows': len(va), 'Latest training outcome': training.Outcome_date.iloc[tr].max(),
                            'First CV date': training.Date.iloc[va].min()}
                           for i, (tr, va) in enumerate(folds)])
display(fold_audit)
search_tables = {}
for name in TUNING_GRIDS:
    search_tables[name] = search_family(name, X_fit, y_fit, folds, fit_candidate)
    print(name, '| configurations:', len(search_tables[name]))
    display(search_tables[name].head(3).round(4))
search_results = pd.concat(search_tables.values(), ignore_index=True)
print('Configurations:', len(search_results), '| CV fits:', len(search_results)*len(folds))

### 15.3 Validation comparison
Refit each family winner on all eligible training observations, then assess the same external validation period used for initial comparisons. No test predictions or final model selection are performed.

In [ ]:
tuned_models = {}
tuned_rows = []
for name, results in search_tables.items():
    best = results.iloc[0]
    model = fit_candidate(name, best['Parameters'], X_fit, y_fit)
    tuned_models[name] = model
    metrics, per_class, matrix = evaluate_predictions(y_val, model.predict(X_val))
    tuned_rows.append({'Model': name + ' (tuned)', **metrics,
                       'Mean CV macro F1': best['Mean CV macro F1'],
                       'SD CV macro F1': best['SD CV macro F1'],
                       'Parameters': best['Parameters']})
    assert matrix.to_numpy().sum() == len(y_val)
    print(name, '| selected configuration:', best['Parameters'])
    display(matrix)
tuned_results = pd.DataFrame(tuned_rows).set_index('Model')
display(tuned_results.round(4))
print(tuned_results.round(4).to_string())
comparison = pd.concat([comparison[baseline_results.columns], tuned_results[baseline_results.columns]])
print(comparison.sort_values('Macro F1', ascending=False).round(4).to_string())
# Export diagnostics only; final model selection and test scoring are separate stages.
search_results.to_csv(export_dir / 'KAIROS_CV_SEARCH.csv', index=False)

## 16. Feature ablation and final model selection
### 16.1 Matched feature sets and selection protocol
Compare seven predictors with a five-predictor price-only set. Keep all observations, chronological folds, targets and metrics identical. Select the final configuration by highest mean training-fold macro F1 across both feature sets; validation scores are diagnostic only. Stable ordering breaks exact ties.

In [ ]:
"""Matched feature-set comparisons and deterministic CV selection."""
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


def feature_sets(columns):
    full = list(columns)
    assert {'VIX_Close', 'VIX_change_1d'}.issubset(full)
    price = [c for c in full if c not in {'VIX_Close', 'VIX_change_1d'}]
    assert len(full) == 7 and len(price) == 5
    return {'With VIX': full, 'Without VIX': price}


def select_by_cv(results):
    assert results['Mean CV macro F1'].notna().all()
    return results.sort_values('Mean CV macro F1', ascending=False, kind='stable').iloc[0]

### 16.2 Fixed-configuration and retuned ablation
Repeat the same 36-configuration search on price-only inputs, using the same folds and fold-local fitting. Extract the full-feature winner configuration from that grid for the fixed comparison.

In [ ]:
sets = feature_sets(FEATURE_COLUMNS)
assert X_fit.index.equals(y_fit.index) and X_val.index.equals(y_val.index)
assert X_fit.index.equals(training.index)
price_X = X_fit[sets['Without VIX']]
price_val = X_val[sets['Without VIX']]
price_search = {}
fixed_rows = []
for name in TUNING_GRIDS:
    full_best = search_tables[name].iloc[0]
    params = full_best['Parameters']
    price_search[name] = search_family(name, price_X, y_fit, folds, fit_candidate)
    # The full-feature winning configuration is also in the price-only grid.
    fixed = price_search[name].loc[price_search[name]['Parameters'].map(lambda p: p == params)].iloc[0]
    fixed_model = fit_candidate(name, params, price_X, y_fit)
    fixed_metrics, _, _ = evaluate_predictions(y_val, fixed_model.predict(price_val))
    fixed_rows.append({'Family': name, 'With VIX CV F1': full_best['Mean CV macro F1'],
                       'Without VIX CV F1': fixed['Mean CV macro F1'],
                       'With VIX validation F1': tuned_results.loc[name + ' (tuned)', 'Macro F1'],
                       'Without VIX validation F1': fixed_metrics['Macro F1']})
    print(name, '| price-only configurations:', len(price_search[name]))
fixed_ablation = pd.DataFrame(fixed_rows).set_index('Family')
display(fixed_ablation.round(4))
print(fixed_ablation.round(4).to_string())

### 16.3 Final configuration lock
Compare the eight family/feature-set winners. Record the CV-selected configuration before final test scoring. The selected estimator is fitted on the original training partition only. Export ablation diagnostics and selection specification.

In [ ]:
ablation_rows = []
for feature_set, columns in sets.items():
    tables = search_tables if feature_set == 'With VIX' else price_search
    for name, table in tables.items():
        best = table.iloc[0]
        model = (tuned_models[name] if feature_set == 'With VIX'
                 else fit_candidate(name, best['Parameters'], X_fit[columns], y_fit))
        metrics, _, matrix = evaluate_predictions(y_val, model.predict(X_val[columns]))
        assert matrix.to_numpy().sum() == len(y_val)
        ablation_rows.append({'Feature set': feature_set, 'Family': name,
                              'Parameters': best['Parameters'],
                              'Mean CV macro F1': best['Mean CV macro F1'],
                              'SD CV macro F1': best['SD CV macro F1'], **metrics})
ablation_results = pd.DataFrame(ablation_rows)
print(ablation_results.round(4).to_string(index=False))
display(ablation_results.round(4))
selected = select_by_cv(ablation_results)
selected_features = sets[selected['Feature set']]
selected_family = selected['Family']
selected_params = dict(selected['Parameters'])
selected_model = fit_candidate(selected_family, selected_params, X_fit[selected_features], y_fit)
assert selected_model.classes_.tolist() == CLASSES
selection_spec = {'family': selected_family, 'feature_set': selected['Feature set'],
                  'features': selected_features, 'parameters': selected_params,
                  'criterion': 'Highest mean training-fold macro F1',
                  'mean_cv_macro_f1': float(selected['Mean CV macro F1'])}
print('LOCKED CONFIGURATION:', selection_spec)
import json
(export_dir / 'KAIROS_SELECTED_MODEL.json').write_text(json.dumps(selection_spec, indent=2))
fixed_ablation.to_csv(export_dir / 'KAIROS_FIXED_ABLATION.csv')
ablation_results.to_csv(export_dir / 'KAIROS_ABLATION_RESULTS.csv', index=False)
pd.concat(price_search.values(), ignore_index=True).to_csv(export_dir / 'KAIROS_PRICE_ONLY_CV_SEARCH.csv', index=False)

## 17. Final test evaluation
### 17.1 Prediction record construction
The selected configuration was recorded before test scoring. The selected forest and the price-only forest use the original training partition only. Model scores are uncalibrated.

In [ ]:
"""Test-period prediction records for a previously selected classifier."""
import numpy as np
import pandas as pd


def prediction_records(model, test, features):
    X_test = test[features]
    assert model.classes_.tolist() == [-1, 0, 1]
    proba = model.predict_proba(X_test)
    pred = model.predict(X_test)
    assert np.isfinite(proba).all() and (proba >= 0).all()
    np.testing.assert_allclose(proba.sum(axis=1), 1)
    np.testing.assert_array_equal(pred, model.classes_[proba.argmax(axis=1)])
    records = test[['Date', 'Outcome_date', 'Forward_return_5d_pct', 'Target']].copy()
    records['Prediction'] = pred
    records[['Score_Down', 'Score_Ordinary', 'Score_Up']] = proba
    records['Correct'] = records.Target.eq(records.Prediction)
    return records

### 17.2 Locked model and baseline comparison
Use identical test observations for the selected forest, the matched price-only forest and two prespecified baselines. Test scores do not alter the model selection.

In [ ]:
# The configuration is fixed before this cell accesses test outcomes.
assert selected_family == 'Forest' and selected['Feature set'] == 'With VIX'
assert selected_params == {'class_weight': 'balanced', 'max_depth': 8, 'min_samples_leaf': 10}
assert selected_features == FEATURE_COLUMNS
assert selected_model.get_params()['n_estimators'] == 200
price_features = sets['Without VIX']
price_params = price_search['Forest'].iloc[0]['Parameters']
price_model = fit_candidate('Forest', price_params, X_fit[price_features], y_fit)
test_data = datasets['test']
assert datasets['train'].Outcome_date.max() < test_data.Date.min()
assert datasets['validation'].Outcome_date.max() < test_data.Date.min()
assert X_fit.index.intersection(test_data.index).empty
X_test = test_data[selected_features].copy()
y_test = test_data['Target'].copy()
assert X_test.index.equals(y_test.index) and X_test.notna().all().all()
records = prediction_records(selected_model, test_data, selected_features)
test_predictions = {'Majority class': majority.predict(X_test),
                    'Momentum': momentum_predict(X_test),
                    'Price-only forest': price_model.predict(X_test[price_features]),
                    'Selected forest': records.Prediction.to_numpy()}
rows = []
for name, pred in test_predictions.items():
    metrics, per_class, matrix = evaluate_predictions(y_test, pred)
    assert matrix.to_numpy().sum() == len(y_test)
    rows.append({'Model': name, **metrics})
    print(name)
    display(per_class.round(4), matrix)
test_results = pd.DataFrame(rows).set_index('Model')
display(test_results.round(4))
print('Test date range:', test_data.Date.min(), 'to', test_data.Date.max())
print('Last outcome date:', test_data.Outcome_date.max())

### 17.3 Error analysis and exports
Plot predictions against observed five-session returns, report directional errors and false alarms, and export test diagnostics.

In [ ]:
# A forecast is scored against its five-session outcome, not a same-day return.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(records.Date, records.Forward_return_5d_pct, color='0.5', linewidth=0.8, label='Actual forward return')
for label, color, name in [(-1, 'tab:red', 'Predicted down'), (1, 'tab:green', 'Predicted up')]:
    mask = records.Prediction.eq(label)
    ax.scatter(records.loc[mask, 'Date'], records.loc[mask, 'Forward_return_5d_pct'], s=18, color=color, label=name)
for threshold in (-3, 3):
    ax.axhline(threshold, linestyle='--', color='0.4', linewidth=0.8)
ax.set(title='Test forecasts and observed five-session returns', xlabel='Forecast date', ylabel='Forward return (%)')
ax.legend(loc='best')
fig.tight_layout()
plt.show()
major = records.Target.ne(0)
print('Major movements correctly classified:', int((major & records.Correct).sum()), '/', int(major.sum()))
print('Ordinary observations flagged as major:', int((~major & records.Prediction.ne(0)).sum()))
print('Major down observations correctly classified:', int((records.Target.eq(-1) & records.Correct).sum()))
records.to_csv(export_dir / 'KAIROS_TEST_PREDICTIONS.csv', index=False)
test_results.to_csv(export_dir / 'KAIROS_TEST_METRICS.csv')

## 18. Version 2: directional forecasting improvement experiments
Retain the Version 1 target, original time partitions and evaluation records. Explore additional causal features, direct versus two-stage classification, class weighting and decision thresholds. Both later periods have already been inspected; comparisons are retrospective. No independent final test is available.

In [ ]:
"""KAIROS V2: causal features and chronological model development."""
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, balanced_accuracy_score, confusion_matrix, average_precision_score
from sklearn.model_selection import TimeSeriesSplit

LABELS = [-1, 0, 1]
V1 = ['Return_1d','Return_5d','Volatility_20d','MA_20_gap_pct','Intraday_range_pct','VIX_Close','VIX_change_1d']


def expanded_features(data, base_features):
    out = base_features.copy()
    c, h, l, o, v = [data[k] for k in ['Close','High','Low','Open','VIX_Close']]
    r = c.pct_change(fill_method=None) * 100
    for n in (3, 10, 20, 60):
        out[f'Return_{n}d'] = c.pct_change(n, fill_method=None) * 100
    for n in (5, 10, 60):
        out[f'Volatility_{n}d'] = r.rolling(n).std()
    out['Vol_ratio_5_20'] = out.Volatility_5d / out.Volatility_20d.replace(0, np.nan)
    out['MA_60_gap_pct'] = (c / c.rolling(60).mean() - 1) * 100
    out['Drawdown_20d'] = (c / c.rolling(20).max() - 1) * 100
    out['Overnight_gap'] = (o / c.shift(1) - 1) * 100
    out['Close_location'] = ((c-l)/(h-l).replace(0, np.nan)).fillna(0.5)
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    out['ATR_14_pct'] = tr.rolling(14).mean() / c * 100
    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    out['RSI_14'] = 100*gain/(gain+loss).replace(0,np.nan)
    out['MACD_pct'] = (c.ewm(span=12, adjust=False).mean()-c.ewm(span=26, adjust=False).mean())/c*100
    # No additional VIX imputation: expanded predictors use only observed price history.
    return out.replace([np.inf,-np.inf],np.nan)


def estimator(kind, weighted):
    weight = 'balanced' if weighted else None
    if kind == 'logistic':
        return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight=weight,max_iter=2000))
    if kind == 'forest':
        return RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=10,max_features='sqrt',class_weight=weight,random_state=42,n_jobs=1)
    if kind == 'extra':
        return ExtraTreesClassifier(n_estimators=120,max_depth=8,min_samples_leaf=10,class_weight=weight,random_state=42,n_jobs=1)
    if kind == 'hist':
        return HistGradientBoostingClassifier(max_iter=80,max_leaf_nodes=7,min_samples_leaf=30,learning_rate=.05,l2_regularization=2,class_weight=weight,early_stopping=False,random_state=42)
    raise ValueError(kind)


def fit_scores(X,y,architecture,kind,weighted):
    if architecture == 'direct':
        model = estimator(kind,weighted).fit(X,y)
        assert list(model.classes_) == LABELS
        return (model,)
    event = estimator(kind,weighted).fit(X,y.ne(0).astype(int))
    direction = estimator(kind,weighted).fit(X.loc[y.ne(0)],y.loc[y.ne(0)].eq(1).astype(int))
    assert list(event.classes_) == list(direction.classes_) == [0,1]
    return event,direction


def scores(models,X):
    if len(models)==1:
        p=models[0].predict_proba(X)
    else:
        m,d=[model.predict_proba(X)[:,1] for model in models]
        p=np.column_stack([m*(1-d),1-m,m*d])
    assert np.isfinite(p).all() and (p>=0).all()
    np.testing.assert_allclose(p.sum(axis=1),1)
    return p


def decide(p,rule):
    if rule[0]=='argmax':
        return np.array(LABELS)[p.argmax(axis=1)]
    _,event_threshold,up_threshold=rule
    event=p[:,0]+p[:,2]
    up=np.divide(p[:,2],event,out=np.full(len(p),.5),where=event>0)
    return np.where(event>=event_threshold,np.where(up>=up_threshold,1,-1),0)


def metrics(y,p):
    y=np.asarray(y); p=np.asarray(p)
    recalls=recall_score(y,p,labels=LABELS,average=None,zero_division=0)
    precisions=precision_score(y,p,labels=LABELS,average=None,zero_division=0)
    return {'Macro F1':f1_score(y,p,labels=LABELS,average='macro',zero_division=0),'Accuracy':accuracy_score(y,p),'Balanced accuracy':balanced_accuracy_score(y,p),'Down recall':recalls[0],'Up recall':recalls[2],'Down precision':precisions[0],'Up precision':precisions[2],'Major correctly classified':int(((y!=0)&(y==p)).sum()),'Ordinary false alarms':int(((y==0)&(p!=0)).sum())}


def search_v2(X,y,training,feature_groups):
    folds=list(TimeSeriesSplit(n_splits=4,test_size=400,gap=5).split(X))
    for tr,va in folds:
        assert training.Outcome_date.iloc[tr].max()<training.Date.iloc[va].min()
        assert set(y.iloc[tr])==set(y.iloc[va])==set(LABELS)
    rules=[('argmax',)] + [('gate',m,d) for m in (.15,.25,.35,.45,.55,.65,.75) for d in (.35,.45,.5,.55,.65)]
    rows=[]
    for group,cols in feature_groups.items():
        for architecture in ('direct','two_stage'):
            for kind in ('logistic','forest','extra','hist'):
                for weighted in (False,True):
                    blocks=[]
                    for tr,va in folds:
                        fitted=fit_scores(X[cols].iloc[tr],y.iloc[tr],architecture,kind,weighted)
                        blocks.append((y.iloc[va],scores(fitted,X[cols].iloc[va])))
                    actual=pd.concat([b[0] for b in blocks])
                    for rule in rules:
                        preds=[decide(p,rule) for _,p in blocks]
                        fold_f1=[f1_score(a,p,labels=LABELS,average='macro',zero_division=0) for (a,_),p in zip(blocks,preds)]
                        pooled=metrics(actual,np.concatenate(preds))
                        rows.append({'features':group,'architecture':architecture,'algorithm':kind,'weighted':weighted,'rule':rule,'Mean CV macro F1':np.mean(fold_f1),'SD CV macro F1':np.std(fold_f1),**{f'Fold {i+1}':f for i,f in enumerate(fold_f1)},**pooled})
    table=pd.DataFrame(rows)
    # Prefer at least 20% recall for both directions in pooled development CV, if feasible.
    table['Recall guardrail']=table[['Down recall','Up recall']].min(axis=1)>=.20
    return table.sort_values(['Recall guardrail','Mean CV macro F1'],ascending=False,kind='stable').reset_index(drop=True)


def risk_metrics(y,p,threshold):
    y=np.asarray(y)!=0; pred=np.asarray(p)>=threshold
    return {'AP':average_precision_score(y,p),'F1':f1_score(y,pred,zero_division=0),
            'Precision':precision_score(y,pred,zero_division=0),'Recall':recall_score(y,pred,zero_division=0),
            'Accuracy':accuracy_score(y,pred),'Detected':int((y&pred).sum()),'False alarms':int((~y&pred).sum()),'Events':int(y.sum())}


def search_risk(X,y,training,feature_groups):
    folds=list(TimeSeriesSplit(n_splits=4,test_size=400,gap=5).split(X))
    for tr,va in folds:
        assert training.Outcome_date.iloc[tr].max()<training.Date.iloc[va].min()
    rows=[]
    for group,cols in feature_groups.items():
        for kind in ('logistic','forest','extra','hist'):
            for weighted in (False,True):
                blocks=[]
                for tr,va in folds:
                    model=estimator(kind,weighted).fit(X[cols].iloc[tr],y.iloc[tr].ne(0).astype(int))
                    blocks.append((y.iloc[va],model.predict_proba(X[cols].iloc[va])[:,1]))
                ap=np.mean([average_precision_score(a.ne(0),p) for a,p in blocks])
                # Pick the warning threshold by fold F1, after imposing pooled recall >= 30% if feasible.
                options=[]
                for threshold in np.arange(.05,.91,.05):
                    m=risk_metrics(pd.concat([a for a,_ in blocks]),np.concatenate([p for _,p in blocks]),threshold)
                    options.append({'threshold':float(threshold),'Mean CV F1':np.mean([risk_metrics(a,p,threshold)['F1'] for a,p in blocks]),**m})
                best=sorted(options,key=lambda q:(q['Recall']>=.30,q['Mean CV F1']),reverse=True)[0]
                rows.append({'features':group,'algorithm':kind,'weighted':weighted,'Mean CV AP':ap,**best})
    return pd.DataFrame(rows).sort_values('Mean CV AP',ascending=False,kind='stable').reset_index(drop=True)

### 18.1 Matched rows and causal feature construction
Compare the original seven features and 22 candidate predictors on identical rows. Longer windows require a 60-session warm-up; the first 40 previously eligible training rows are excluded.

In [ ]:
# Preserve all V1 outputs and restrict V2 comparisons to matched eligible rows.
Z_v2 = expanded_features(df, X)
for cutoff in (1500, train_end, val_end):
    pd.testing.assert_frame_equal(Z_v2.iloc[:cutoff], expanded_features(df.iloc[:cutoff], X.iloc[:cutoff]))
parts_v2 = {}
for name, part in datasets.items():
    new = part.set_index('Date').join(Z_v2.set_index(df.Date).drop(columns=V1), how='left')
    parts_v2[name] = new.dropna(subset=Z_v2.columns).reset_index()
    assert parts_v2[name].Date.is_monotonic_increasing
    assert parts_v2[name][Z_v2.columns].notna().all().all()
    print(name, len(parts_v2[name]), parts_v2[name].Target.value_counts().sort_index().to_dict())
for name in ('validation', 'test'):
    assert parts_v2[name].Date.tolist() == datasets[name].Date.tolist()
training_v2 = parts_v2['train']
groups_v2 = {'original': V1, 'expanded': Z_v2.columns.tolist()}
print('Candidate features:', groups_v2)

### 18.2 Development-only model and decision-rule selection
Use four expanding-window folds with a five-row gap and outcome-date checks. Rank by mean fold macro F1, preferring configurations with pooled recall of at least 20% for both directional classes. Decision thresholds are optimized on these same folds, so CV scores are selection scores, not independent estimates.

In [ ]:
from threadpoolctl import threadpool_limits
import json
from pathlib import Path
import joblib
import sklearn

v2_dir = export_dir / 'v2'
v2_dir.mkdir(parents=True, exist_ok=True)
with threadpool_limits(limits=1):
    v2_search = search_v2(training_v2[Z_v2.columns], training_v2.Target, training_v2, groups_v2)
    best_v2 = v2_search.iloc[0]
    v2_features = groups_v2[best_v2.features]
    v2_models = fit_scores(training_v2[v2_features], training_v2.Target, best_v2.architecture, best_v2.algorithm, bool(best_v2.weighted))
    v2_rule = best_v2.rule
v2_spec = {'features': v2_features, 'feature_set': best_v2.features, 'architecture': best_v2.architecture,
           'algorithm': best_v2.algorithm, 'weighted': bool(best_v2.weighted), 'rule': list(v2_rule),
           'mean_cv_macro_f1': float(best_v2['Mean CV macro F1']), 'recall_guardrail': bool(best_v2['Recall guardrail'])}
(v2_dir / 'selection.json').write_text(json.dumps(v2_spec, indent=2))
v2_search.to_csv(v2_dir / 'search.csv', index=False)
print('Selected specification:', v2_spec)
display(v2_search.head(10).round(4))

### 18.3 Retrospective directional evaluation and packaging
Compare V2 with the previously locked V1 forest on identical dates. No model is promoted solely because of test performance. All scores are uncalibrated.

In [ ]:
# Both later periods were inspected during V1; these are retrospective comparisons, not new holdouts.
v2_rows = []
for name in ('validation', 'test'):
    part = parts_v2[name]
    assert training_v2.Outcome_date.max() < part.Date.min()
    actual = part.Target
    p = scores(v2_models, part[v2_features])
    pred = decide(p, v2_rule)
    old_pred = selected_model.predict(part[V1])
    old_p = selected_model.predict_proba(part[V1])
    for label, predictions_v2, probabilities_v2 in [('V1', old_pred, old_p), ('V2', pred, p)]:
        v2_rows.append({'Period': name, 'Model': label, **metrics(actual, predictions_v2),
                        'Major AP': average_precision_score(actual.ne(0), probabilities_v2[:,0]+probabilities_v2[:,2])})
        print(name, label)
        display(pd.DataFrame(confusion_matrix(actual, predictions_v2, labels=LABELS), index=LABELS, columns=LABELS))
    rec = part[['Date','Outcome_date','Forward_return_5d_pct','Target']].copy()
    rec['Prediction'] = pred
    rec[['Score_Down','Score_Ordinary','Score_Up']] = p
    rec['V1_Prediction'] = old_pred
    rec.to_csv(v2_dir / f'{name}_predictions.csv', index=False)
v2_metrics = pd.DataFrame(v2_rows)
display(v2_metrics.round(4))
v2_metrics.to_csv(v2_dir / 'metrics.csv', index=False)
# A fitted research model is packaged with the feature and decision schema; scores are not calibrated.
bundle_v2 = {'models': v2_models, 'spec': v2_spec, 'sklearn_version': sklearn.__version__,
             'training_last_outcome': str(training_v2.Outcome_date.max().date())}
joblib.dump(bundle_v2, v2_dir / 'KAIROS_V2.joblib')
reloaded = joblib.load(v2_dir / 'KAIROS_V2.joblib')
np.testing.assert_array_equal(decide(scores(reloaded['models'], parts_v2['test'][v2_features]), reloaded['spec']['rule']), pred)
latest = df[['Date']].join(Z_v2).tail(5).dropna(subset=v2_features).copy()
latest_p = scores(v2_models, latest[v2_features])
latest['Prediction'] = decide(latest_p, v2_rule)
latest[['Score_Down','Score_Ordinary','Score_Up']] = latest_p
latest[['Date','Prediction','Score_Down','Score_Ordinary','Score_Up']].to_csv(v2_dir / 'latest_snapshot_predictions.csv', index=False)
print('Model artifact and historical snapshot predictions saved to', v2_dir)

## 19. Auxiliary significant-movement risk model
Separate the magnitude question (absolute five-session return greater than 3%) from the three-class direction task. Select by mean training-fold average precision, and set a warning threshold using training-fold F1. Evaluate on already examined periods, comparing against event prevalence and a VIX ranking baseline. Binary risk results are not directly comparable with multiclass F1.

In [ ]:
# Auxiliary task: detect magnitude, not direction. It is not a replacement score for three-class forecasts.
with threadpool_limits(limits=1):
    risk_search = search_risk(training_v2[Z_v2.columns], training_v2.Target, training_v2, groups_v2)
    risk_best = risk_search.iloc[0]
    risk_features = groups_v2[risk_best.features]
    risk_model = estimator(risk_best.algorithm, bool(risk_best.weighted)).fit(training_v2[risk_features], training_v2.Target.ne(0).astype(int))
risk_threshold = float(risk_best.threshold)
risk_spec = {'features': risk_features, 'feature_set': risk_best.features, 'algorithm': risk_best.algorithm,
             'weighted': bool(risk_best.weighted), 'threshold': risk_threshold, 'mean_cv_ap': float(risk_best['Mean CV AP'])}
(v2_dir / 'risk_selection.json').write_text(json.dumps(risk_spec, indent=2))
risk_search.to_csv(v2_dir / 'risk_search.csv', index=False)
print('Auxiliary selection:', risk_spec)
display(risk_search.round(4))
risk_rows = []
for name in ('validation', 'test'):
    part = parts_v2[name]
    p = risk_model.predict_proba(part[risk_features])[:,1]
    for label, score, cutoff in [('Risk model', p, risk_threshold), ('Always ordinary', np.zeros(len(part)), .5), ('VIX rank baseline', part.VIX_Close.to_numpy(), np.inf)]:
        result = risk_metrics(part.Target, score, cutoff)
        if label == 'VIX rank baseline':
            result = {'AP':result['AP'], 'Events':result['Events']}
        risk_rows.append({'Period':name,'Model':label, 'Event prevalence':part.Target.ne(0).mean(), **result})
    rec = part[['Date','Outcome_date','Forward_return_5d_pct','Target']].copy()
    rec['Risk_score'] = p
    rec['Warning'] = p>=risk_threshold
    rec.to_csv(v2_dir / f'{name}_risk_predictions.csv', index=False)
risk_results = pd.DataFrame(risk_rows)
display(risk_results.round(4))
risk_results.to_csv(v2_dir / 'risk_metrics.csv', index=False)
bundle_risk = {'model':risk_model,'spec':risk_spec,'sklearn_version':sklearn.__version__, 'training_last_outcome':str(training_v2.Outcome_date.max().date())}
joblib.dump(bundle_risk, v2_dir / 'KAIROS_RISK.joblib')
reloaded_risk = joblib.load(v2_dir / 'KAIROS_RISK.joblib')
np.testing.assert_allclose(reloaded_risk['model'].predict_proba(parts_v2['test'][risk_features]), risk_model.predict_proba(parts_v2['test'][risk_features]))
latest_risk = df[['Date']].join(Z_v2).tail(5).dropna(subset=risk_features).copy()
latest_risk['Risk_score'] = risk_model.predict_proba(latest_risk[risk_features])[:,1]
latest_risk['Warning'] = latest_risk.Risk_score>=risk_threshold
latest_risk[['Date','Risk_score','Warning']].to_csv(v2_dir / 'latest_snapshot_risk.csv', index=False)

## 20. Institutional cash-flow extension (pending data)
This optional experiment compares the original seven inputs against 15 inputs (eight FII/DII flow features) on identical eligible rows with fixed forest hyperparameters. Daily combined-exchange cash-market data must be supplied as KAIROS_FII.csv and KAIROS_DII.csv, each with ISO Date, Buy, Sell and Net (INR crore). One-session lag assumes provisional figures are published before the following market close; verify source timing and revisions before research use. Missing flows are not imputed. The 80% coverage gate is a preliminary quality criterion, not proof of representative sampling. Both later periods remain retrospective. No actual institutional flow dataset has been obtained or evaluated.

In [ ]:
"""Cash-market FII/DII features and matched retrospective evaluation."""
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, recall_score
from sklearn.model_selection import TimeSeriesSplit

FLOW_COLUMNS = [f'{g}_{s}' for g in ('FII','DII') for s in ('net_lag1','imbalance_lag1','imbalance_5d','imbalance_20d')]


def prepare_flow(frame, sessions, group):
    """Input: Date, Buy, Sell, Net (INR crore); align to previous market session."""
    if group not in ('FII', 'DII'):
        raise ValueError('Invalid investor category')
    f = frame[['Date','Buy','Sell','Net']].copy()
    f.Date = pd.to_datetime(f.Date, errors='raise').dt.normalize()
    if f.Date.isna().any() or f.Date.duplicated().any():
        raise ValueError('Missing or duplicate flow dates')
    for col in ('Buy','Sell','Net'):
        f[col] = pd.to_numeric(f[col].astype(str).str.replace(',','',regex=False), errors='raise')
    if not np.isfinite(f[['Buy','Sell','Net']]).all().all() or (f[['Buy','Sell']] < 0).any().any():
        raise ValueError('Invalid flow amounts')
    if not np.allclose(f.Buy-f.Sell, f.Net, atol=.05, rtol=0):
        raise ValueError('Net must equal Buy minus Sell (rounding tolerance 0.05 crore)')
    dates = pd.DatetimeIndex(pd.to_datetime(sessions), name='Date')
    if not dates.is_unique or not dates.is_monotonic_increasing:
        raise ValueError('Market sessions must be unique and chronological')
    f = f.set_index('Date').reindex(dates)
    # Never impute missing daily flows or use same-session post-close releases.
    gross = f.Buy + f.Sell
    imbalance = f.Net / gross.replace(0,np.nan)
    out = pd.DataFrame({f'{group}_net_lag1':f.Net.shift(1),
                        f'{group}_imbalance_lag1':imbalance.shift(1),
                        f'{group}_imbalance_5d':(f.Net.rolling(5).sum()/gross.rolling(5).sum().replace(0,np.nan)).shift(1),
                        f'{group}_imbalance_20d':(f.Net.rolling(20).sum()/gross.rolling(20).sum().replace(0,np.nan)).shift(1)}, index=dates)
    return out.replace([np.inf,-np.inf],np.nan)


def compare_flows(datasets, flow_features, original_features):
    """Same forecast dates and model settings; only inputs differ."""
    matched, audit = {}, []
    for name, part in datasets.items():
        merged = part.join(flow_features, on='Date', validate='one_to_one')
        matched[name] = merged.dropna(subset=FLOW_COLUMNS).reset_index(drop=True)
        audit.append({'Partition':name,'Original rows':len(part),'Matched rows':len(matched[name]),'Coverage':len(matched[name])/len(part)})
    audit = pd.DataFrame(audit)
    if (audit.Coverage < .8).any():
        raise ValueError('Insufficient coverage: '+audit.to_string(index=False))
    train = matched['train']
    folds = list(TimeSeriesSplit(n_splits=4,test_size=400,gap=5).split(train))
    for tr, va in folds:
        if train.Outcome_date.iloc[tr].max() >= train.Date.iloc[va].min():
            raise ValueError('Overlapping outcomes')
        if set(train.Target.iloc[tr]) != {-1,0,1} or set(train.Target.iloc[va]) != {-1,0,1}:
            raise ValueError('Missing class in fold')
    results, models = [], {}
    for name, cols in {'Original':list(original_features),'FII+DII':list(original_features)+FLOW_COLUMNS}.items():
        def build():
            return RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=10,criterion='gini',max_features='sqrt',bootstrap=True,class_weight='balanced',random_state=42,n_jobs=1)
        cv = [f1_score(train.Target.iloc[va],build().fit(train[cols].iloc[tr],train.Target.iloc[tr]).predict(train[cols].iloc[va]),labels=[-1,0,1],average='macro',zero_division=0) for tr,va in folds]
        model = build().fit(train[cols],train.Target)
        models[name] = model
        for period in ('validation','test'):
            part=matched[period]
            if train.Outcome_date.max() >= part.Date.min():
                raise ValueError('Training outcomes overlap evaluation')
            pred=model.predict(part[cols]); y=part.Target
            recalls=recall_score(y,pred,labels=[-1,0,1],average=None,zero_division=0)
            results.append({'Model':name,'Period':period,'Rows':len(y),'Mean CV macro F1':np.mean(cv),'Macro F1':f1_score(y,pred,labels=[-1,0,1],average='macro',zero_division=0),'Accuracy':accuracy_score(y,pred),'Balanced accuracy':balanced_accuracy_score(y,pred),'Down recall':recalls[0],'Up recall':recalls[2],'Major correct':int(((y!=0)&(y==pred)).sum()),'Ordinary false alarms':int(((y==0)&(pred!=0)).sum())})
    return pd.DataFrame(results), audit, models

In [ ]:
fii_path = base / 'KAIROS_FII.csv'
dii_path = base / 'KAIROS_DII.csv'
if not (fii_path.is_file() and dii_path.is_file()):
    print('FII/DII experiment pending: supply both daily cash-market CSVs (Date, Buy, Sell, Net; INR crore). No performance result has been calculated.')
else:
    fii = prepare_flow(pd.read_csv(fii_path), df.Date, 'FII')
    dii = prepare_flow(pd.read_csv(dii_path), df.Date, 'DII')
    flow_results, flow_audit, flow_models = compare_flows(datasets, fii.join(dii), FEATURE_COLUMNS)
    display(flow_audit, flow_results.round(4))
    flow_results.to_csv(export_dir / 'KAIROS_FLOW_COMPARISON.csv', index=False)
    flow_audit.to_csv(export_dir / 'KAIROS_FLOW_COVERAGE.csv', index=False)

## 21. Preliminary monthly FII/DII experiment
A secondary monthly archive (HARI-KIRAN-REDDY/FIIDIIData, fii_dii.csv, blob 325849df90ab0ce25dba3b69ed67de6258393076) supplies January 2010–December 2025 net flows in INR crore. The snapshot excludes the partial January 2026 month. Use the month two calendar months before the forecast date, assuming publication by then; exact historical release timing and revisions are unverified. No carry-forward beyond the archive. Compare original, FII only, DII only and both with fixed forest settings and identical forecast dates. Validation and historical test periods have already been inspected. Test comparisons use 434 rows, 4 June 2024–27 February 2026.

In [ ]:
"""Preliminary monthly institutional flow comparison."""
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, recall_score, confusion_matrix, average_precision_score
from sklearn.model_selection import TimeSeriesSplit

MONTHLY_COLS = {g: [f'{g}_monthly_net', f'{g}_three_month_net'] for g in ('FII','DII')}

def monthly_features(monthly, dates):
    m=monthly[['Month','FII_net','DII_net']].copy()
    m.index=pd.PeriodIndex(m.pop('Month'),freq='M')
    m=m.sort_index().astype(float)
    assert m.index.is_unique and m.index.equals(pd.period_range(m.index.min(),m.index.max(),freq='M'))
    assert np.isfinite(m.to_numpy()).all()
    z=pd.DataFrame(index=m.index)
    for g in ('FII','DII'):
        z[MONTHLY_COLS[g][0]]=m[g+'_net']
        z[MONTHLY_COLS[g][1]]=m[g+'_net'].rolling(3).sum()
    # Use M-2 for forecast month M. Exact match, no indefinite carry-forward.
    d=pd.DatetimeIndex(pd.to_datetime(dates),name='Date')
    keys=d.to_period('M')-2
    out=z.reindex(keys).set_axis(d)
    out['Flow_month']=keys.astype(str)
    return out


def monthly_comparison(datasets, features, original_features):
    cols=MONTHLY_COLS['FII']+MONTHLY_COLS['DII']
    matched={}; audit=[]
    for period,part in datasets.items():
        part=part.copy(); part['Date']=pd.to_datetime(part.Date); part['Outcome_date']=pd.to_datetime(part.Outcome_date)
        assert part.Date.is_unique and part.Date.is_monotonic_increasing
        frame=part.join(features,on='Date',validate='one_to_one').dropna(subset=cols).reset_index(drop=True)
        matched[period]=frame
        audit.append({'Period':period,'Original rows':len(part),'Matched rows':len(frame),'Start':frame.Date.min(),'End':frame.Date.max(),'Major Down':int(frame.Target.eq(-1).sum()),'Ordinary':int(frame.Target.eq(0).sum()),'Major Up':int(frame.Target.eq(1).sum())})
    train=matched['train']
    folds=list(TimeSeriesSplit(n_splits=4,test_size=400,gap=5).split(train))
    for tr,va in folds:
        assert train.Outcome_date.iloc[tr].max()<train.Date.iloc[va].min()
        assert set(train.Target.iloc[tr])==set(train.Target.iloc[va])=={-1,0,1}
    def build():
        return RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=10,criterion='gini',max_features='sqrt',bootstrap=True,class_weight='balanced',random_state=42,n_jobs=1)
    rows=[]; preds=[]; fold_rows=[]
    for name,extra in [('Original',[]),('FII only',MONTHLY_COLS['FII']),('DII only',MONTHLY_COLS['DII']),('FII+DII',cols)]:
        c=list(original_features)+extra
        cv=[]
        for k,(tr,va) in enumerate(folds,1):
            model=build().fit(train[c].iloc[tr],train.Target.iloc[tr]); pred=model.predict(train[c].iloc[va]); score=f1_score(train.Target.iloc[va],pred,labels=[-1,0,1],average='macro',zero_division=0)
            cv.append(score); fold_rows.append({'Model':name,'Fold':k,'Macro F1':score})
        model=build().fit(train[c],train.Target)
        for period in ('validation','test'):
            p=matched[period]; assert train.Outcome_date.max()<p.Date.min(); y=p.Target; pred=model.predict(p[c]); prob=model.predict_proba(p[c]); assert model.classes_.tolist()==[-1,0,1]
            rec=recall_score(y,pred,labels=[-1,0,1],average=None,zero_division=0); matrix=confusion_matrix(y,pred,labels=[-1,0,1])
            rows.append({'Model':name,'Period':period,'Rows':len(y),'Mean CV macro F1':np.mean(cv),'Macro F1':f1_score(y,pred,labels=[-1,0,1],average='macro',zero_division=0),'Accuracy':accuracy_score(y,pred),'Balanced accuracy':balanced_accuracy_score(y,pred),'Down recall':rec[0],'Up recall':rec[2],'Major correct':int(((y!=0)&(y==pred)).sum()),'Ordinary false alarms':int(((y==0)&(pred!=0)).sum()),'Event AP':average_precision_score(y.ne(0),prob[:,0]+prob[:,2]),'Confusion matrix':str(matrix.tolist())})
            records=p[['Date','Outcome_date','Target','Flow_month']].copy(); records['Model']=name; records['Period']=period; records['Prediction']=pred; preds.append(records)
    return pd.DataFrame(rows),pd.DataFrame(audit),pd.concat(preds,ignore_index=True),pd.DataFrame(fold_rows)


In [ ]:
monthly_path = base / 'KAIROS_MONTHLY_FLOWS.csv'
if monthly_path.is_file():
    monthly = pd.read_csv(monthly_path)
    monthly_dates = pd.concat([part.Date for part in datasets.values()])
    monthly_z = monthly_features(monthly, monthly_dates)
    cutoff = pd.Timestamp('2021-01-15')
    pd.testing.assert_frame_equal(monthly_z.loc[monthly_z.index <= cutoff],
        monthly_features(monthly[monthly.Month <= '2020-11'], monthly_dates[monthly_dates <= cutoff]))
    monthly_results, monthly_audit, monthly_predictions, monthly_folds = monthly_comparison(datasets, monthly_z, FEATURE_COLUMNS)
    display(monthly_audit, monthly_results.round(4), monthly_folds.round(4))
    for name, frame in [('comparison',monthly_results),('coverage',monthly_audit),('predictions',monthly_predictions),('folds',monthly_folds)]:
        frame.to_csv(export_dir / f'KAIROS_MONTHLY_{name.upper()}.csv', index=False)
else:
    print('Monthly flow experiment requires KAIROS_MONTHLY_FLOWS.csv alongside the original raw CSV files.')

## 22. Final model consolidation
Preserve the original locked directional forest as reference (stronger validation macro F1), the monthly FII-only CV-leading candidate and the four-dataset forest as experimental alternatives. The risk model remains auxiliary. No further test-driven tuning. Refit directional models on original eligible training rows, reproduce matched metrics, export a single versioned bundle and verify identical reloaded outputs. Monthly model inputs require an available month two calendar months before the forecast date. Scores are uncalibrated; feature-ready input values must be derived causally from the supplied data.

In [ ]:
"""Freeze the reference model and experimental flow candidates without test tuning."""
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, recall_score

BASE = ['Return_1d','Return_5d','Volatility_20d','MA_20_gap_pct','Intraday_range_pct','VIX_Close','VIX_change_1d']
FII = ['FII_monthly_net','FII_three_month_net']
DII = ['DII_monthly_net','DII_three_month_net']
FEATURES = {'reference': BASE, 'fii': BASE+FII, 'all_four': BASE+FII+DII}
LABELS = [-1,0,1]


def freeze_models(datasets, monthly_z, risk_model, risk_spec):
    train = datasets['train'].copy()
    matched = train.join(monthly_z, on='Date', validate='one_to_one').dropna(subset=FII+DII)
    assert train.Date.is_unique and train.Date.is_monotonic_increasing
    assert train.Outcome_date.max() < datasets['validation'].Date.min()
    assert train.Outcome_date.max() < datasets['test'].Date.min()
    models = {}
    for name, cols in FEATURES.items():
        part = train if name == 'reference' else matched
        assert part[cols].notna().all().all()
        model = RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=10,criterion='gini',max_features='sqrt',bootstrap=True,class_weight='balanced',random_state=42,n_jobs=1)
        model.fit(part[cols],part.Target)
        assert model.classes_.tolist() == LABELS
        models[name] = model
    assert list(risk_spec['features']) == BASE and list(risk_model.classes_) == [0,1]
    return {'models':models,'features':FEATURES,'risk_model':risk_model,'risk_threshold':float(risk_spec['threshold']),
            'roles':{'reference':'Directional reference; original locked V1','fii':'Experimental CV-leading monthly-flow candidate','all_four':'Experimental four-dataset candidate'},
            'sklearn_version':sklearn.__version__,'training_rows':{'reference':len(train),'fii':len(matched),'all_four':len(matched)},
            'training_last_outcome':str(train.Outcome_date.max().date()),'flow_months':sorted(monthly_z.dropna(subset=FII+DII).Flow_month.unique().tolist()),
            'horizon':5,'threshold_pct':3.0}


def predict_frame(bundle, frame, name='reference'):
    if name not in bundle['models']:
        raise ValueError('Unknown model')
    cols = bundle['features'][name]
    if not set(['Date']+cols).issubset(frame.columns):
        raise ValueError('Missing required input columns')
    dates = pd.to_datetime(frame.Date, errors='raise')
    if dates.isna().any() or dates.duplicated().any() or not dates.is_monotonic_increasing:
        raise ValueError('Dates must be unique and chronological')
    if name != 'reference':
        if 'Flow_month' not in frame:
            raise ValueError('Flow_month is required for institutional models')
        expected = (dates.dt.to_period('M')-2).astype(str)
        if not frame.Flow_month.astype(str).eq(expected).all() or not expected.isin(bundle['flow_months']).all():
            raise ValueError('Unavailable or wrongly lagged flow month')
    if not np.isfinite(frame[cols].to_numpy(dtype=float)).all():
        raise ValueError('Nonfinite inputs')
    model=bundle['models'][name]
    prob=model.predict_proba(frame[cols])
    np.testing.assert_allclose(prob.sum(axis=1),1)
    risk=bundle['risk_model'].predict_proba(frame[BASE])[:,1]
    result=pd.DataFrame({'Date':dates,'Model':name,'Prediction':model.predict(frame[cols]),'Score_Down':prob[:,0],'Score_Ordinary':prob[:,1],'Score_Up':prob[:,2],'Risk_score':risk,'Risk_warning':risk>=bundle['risk_threshold']},index=frame.index)
    assert np.array_equal(result.Prediction.to_numpy(),np.array(LABELS)[prob.argmax(axis=1)])
    return result


def consolidate_metrics(bundle, datasets, monthly_z):
    rows=[]; predictions=[]
    for period in ('validation','test'):
        p=datasets[period].copy()
        matched=p.join(monthly_z,on='Date',validate='one_to_one').dropna(subset=FII+DII)
        for scope, part, names in [('full',p,['reference']),('matched',matched,list(FEATURES))]:
            for name in names:
                pred=predict_frame(bundle,part,name)
                y=part.Target
                r=recall_score(y,pred.Prediction,labels=LABELS,average=None,zero_division=0)
                rows.append({'Period':period,'Scope':scope,'Model':name,'Rows':len(y),'Accuracy':accuracy_score(y,pred.Prediction),'Macro F1':f1_score(y,pred.Prediction,labels=LABELS,average='macro',zero_division=0),'Balanced accuracy':balanced_accuracy_score(y,pred.Prediction),'Down recall':r[0],'Up recall':r[2]})
                pred['Target']=y; pred['Outcome_date']=part.Outcome_date; pred['Forward_return_5d_pct']=part.Forward_return_5d_pct; pred['Scope']=scope; pred['Period']=period; predictions.append(pred)
    return pd.DataFrame(rows),pd.concat(predictions,ignore_index=True)

In [ ]:
# Freeze the existing model roles; do not optimize on previously inspected test dates.
if not (base / 'KAIROS_MONTHLY_FLOWS.csv').is_file():
    print('Final consolidation requires KAIROS_MONTHLY_FLOWS.csv alongside the original sources.')
else:
    final_bundle = freeze_models(datasets, monthly_z, risk_model, risk_spec)
    final_metrics, final_predictions = consolidate_metrics(final_bundle, datasets, monthly_z)
    for _, row in final_metrics.query("Scope == 'matched'").iterrows():
        label = {'reference':'Original','fii':'FII only','all_four':'FII+DII'}[row.Model]
        prior = monthly_results[(monthly_results.Model == label) & (monthly_results.Period == row.Period)].iloc[0]
        keys = ['Accuracy','Macro F1','Balanced accuracy','Down recall','Up recall']
        np.testing.assert_allclose(row[keys].astype(float), prior[keys].astype(float), atol=1e-12)
    out_final = export_dir / 'final'
    out_final.mkdir(exist_ok=True, parents=True)
    joblib.dump(final_bundle, out_final / 'KAIROS_FINAL.joblib', compress=3)
    final_loaded = joblib.load(out_final / 'KAIROS_FINAL.joblib')
    check_metrics, check_predictions = consolidate_metrics(final_loaded, datasets, monthly_z)
    pd.testing.assert_frame_equal(final_metrics, check_metrics)
    pd.testing.assert_frame_equal(final_predictions, check_predictions)
    final_metrics.to_csv(out_final / 'KAIROS_FINAL_METRICS.csv', index=False)
    final_predictions.to_csv(out_final / 'KAIROS_FINAL_PREDICTIONS.csv', index=False)
    manifest = {k:v for k,v in final_bundle.items() if k not in ('models','risk_model')}
    (out_final / 'KAIROS_MODEL_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
    display(final_metrics.round(4))
    print('Verified model bundle saved:', out_final)

## 23. Scope and limitations
The selected configuration was locked by mean chronological training-fold macro F1 before test scoring. The historical snapshot has been examined in earlier project versions, so the test period is a procedural holdout, not independent external validation. Overlapping five-session outcomes are statistically dependent; scores describe observations, not independent market episodes. Class probabilities are uncalibrated. The selected model failed to classify any major declines correctly in the test period. No trading profitability or reliable future forecasting is established. Live acquisition has not been retested. Version 2 and auxiliary magnitude experiments are retrospective because the historical test period has already been inspected. Neither proves independent forecasting performance; new data are necessary. V1 remains the original benchmark. The monthly flow experiment uses secondary historical totals, not verified point-in-time daily reports. Its shorter retrospective test window excludes March–November 2026. No new independent test or directional model promotion is claimed. Final consolidation preserves V1 as the reference, the FII-only CV-leading candidate and all-four candidate as exploratory, and the separate binary risk model. The bundle is not intended for trading.